# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# --- Setup ---
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
url = "hf://datasets/FlyRank/internship-warehouse"
rel = f"{url}/fact_content_daily_performance/**/*.parquet"
print("Connected!")

Connected!


In [2]:
T = "2026-02-01"  # validated cutoff -- 39 eligible clients

WINDOW_DAYS = 30
LABEL_GAP_DAYS = 1
LABEL_WINDOW_DAYS = 30

# --- Feature set, as of T only ---
content_type_query = f"""
    SELECT content_hash_id, keyword_char_count, url_char_count, content_type,
        search_volume, competition, main_intent, category_count, model_used, char_count,
        DATEDIFF('day', content_updated_date, DATE '{T}') AS days_since_last_update,
        DATEDIFF('day', content_created_date, DATE '{T}') AS days_since_created
    FROM read_parquet('{url}/dim_content.parquet')
"""
content_dim = con.sql(content_type_query).df()

feature_query = f"""
    SELECT client_hash_id, content_hash_id,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{rel}')
    WHERE report_date <= DATE '{T}'
    GROUP BY client_hash_id, content_hash_id
"""
X_raw = con.sql(feature_query).df().merge(content_dim, on="content_hash_id", how="left")

# --- Gate: is_declining at T ---
gate_query = f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {WINDOW_DAYS - 1} DAY) AND DATE '{T}'
                     THEN gsc_impressions ELSE 0 END) AS impr_last30,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {2*WINDOW_DAYS - 1} DAY)
                                           AND (DATE '{T}' - INTERVAL {WINDOW_DAYS} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_prev30
        FROM read_parquet('{rel}')
        WHERE report_date <= DATE '{T}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN (impr_last30 - impr_prev30) / NULLIF(impr_prev30, 0) * 100 < -20
             THEN 1 ELSE 0 END AS is_declining_at_T
    FROM windowed
"""
gate_df = con.sql(gate_query).df()

# --- Recovery label: strictly after T ---
future_query = f"""
    WITH future AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_T1,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_baseline
        FROM read_parquet('{rel}')
        WHERE report_date > DATE '{T}'
          AND report_date <= (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN impr_T1 > impr_baseline THEN 1 ELSE 0 END AS recovered_by_T1
    FROM future
"""
recovery_df = con.sql(future_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
# --- Impute missing data ---
X_raw["has_keyword_data"] = X_raw["search_volume"].notna() & X_raw["competition"].notna()
X_raw["char_count_imputed"] = X_raw["char_count"].isna()
X_raw["ai_generated"] = X_raw["model_used"].notna()

X_raw["gsc_avg_position"] = X_raw["gsc_avg_position"].fillna(0)
X_raw["search_volume"] = X_raw["search_volume"].fillna(0)
X_raw["competition"] = X_raw["competition"].fillna(0)
X_raw["main_intent"] = X_raw["main_intent"].fillna("no_keyword")
X_raw["model_used"] = X_raw["model_used"].fillna("human")
X_raw["char_count"] = X_raw["char_count"].fillna(X_raw["char_count"].median())

# --- Assemble data, y, groups ---
declining_ids = gate_df.loc[gate_df["is_declining_at_T"] == 1, ["client_hash_id", "content_hash_id"]]

data = (X_raw.merge(declining_ids, on=["client_hash_id", "content_hash_id"], how="inner")
             .merge(recovery_df[["client_hash_id", "content_hash_id", "recovered_by_T1"]],
                     on=["client_hash_id", "content_hash_id"], how="inner"))

y = data.pop("recovered_by_T1")
groups = data["client_hash_id"]
categorical_text_cols = ["content_type", "main_intent", "model_used"]

# --- Build R2: drop confound cluster, add ctr, encode without model_used ---
drop_r2 = ["char_count", "char_count_imputed", "keyword_char_count", "gsc_clicks", "model_used", "ai_generated"]
categorical_r2 = [c for c in categorical_text_cols if c != "model_used"]

X_r2 = pd.get_dummies(
    data.drop(columns=["client_hash_id", "content_hash_id"] +
              [c for c in drop_r2 if c in data.columns]),
    columns=categorical_r2
)

X_r2["ctr"] = data["gsc_clicks"] / data["gsc_impressions"].replace(0, np.nan)
X_r2["ctr"] = X_r2["ctr"].fillna(0)

print(f"X_r2: {X_r2.shape[0]} rows, {X_r2.shape[1]} features, {groups.nunique()} clients")
print(f"y positive rate: {y.mean():.3f}")

X_r2: 34402 rows, 17 features, 38 clients
y positive rate: 0.402


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf_final = RandomForestClassifier(
    max_depth=4,
    random_state=42,
    class_weight="balanced"
)
rf_final.fit(X_r2, y)

print(f"Trained on {len(X_r2)} rows, {X_r2.shape[1]} features")

Trained on 34402 rows, 17 features


In [5]:
# --- P(recovery) from the trained model, for every declining page ---
p_recovery = rf_final.predict_proba(X_r2)[:, 1]

# --- impact_at_risk: how much is at stake, using decline magnitude x scale ---
# trend_pct at T (the gate's own computation) + impressions, both real/observed, no leakage
# rebuild trend_pct_at_T from the same windowed query used for the gate
trend_query = f"""
    SELECT client_hash_id, content_hash_id,
        (impr_last30 - impr_prev30) / NULLIF(impr_prev30, 0) * 100 AS trend_pct_at_T
    FROM (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {WINDOW_DAYS - 1} DAY) AND DATE '{T}'
                     THEN gsc_impressions ELSE 0 END) AS impr_last30,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {2*WINDOW_DAYS - 1} DAY)
                                           AND (DATE '{T}' - INTERVAL {WINDOW_DAYS} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_prev30
        FROM read_parquet('{rel}')
        WHERE report_date <= DATE '{T}'
        GROUP BY client_hash_id, content_hash_id
    )
"""
trend_df = con.sql(trend_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                content_hash_id  priority_score  impact_at_risk  p_recovery
9473   content_5519ab199297bdb2        0.527189        0.720535    0.268336
25965  content_469512b149928d3e        0.471043        0.859845    0.452177
26973  content_972690231ea8c69f        0.466190        0.598538    0.221118
3885   content_eb77135ddda07ea8        0.460199        0.567226    0.188685
26836  content_c9a819d1b9ad7420        0.456771        0.627437    0.272005
33547  content_eea3b2434e98edf3        0.455715        0.667077    0.316848
26690  content_fcc1f8ce43abd284        0.451740        0.580758    0.222154
14350  content_6945124cd32196d5        0.448963        0.514359    0.127141
16303  content_4e1a6900719d2ef1        0.441381        0.673965    0.345098
27195  content_0f98153d0219fd53        0.439877        0.502724    0.125011
9468   content_e058dee5b52e4527        0.432610        0.577110    0.250386
16308  content_53031986a96b7913        0.430397        0.674327    0.361738
5160   conte

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [16]:
# join trend_pct onto your declining-page population (same rows as X_r2/y, in order)
scored = data[["client_hash_id", "content_hash_id"]].merge(
    trend_df, on=["client_hash_id", "content_hash_id"], how="left"
)

# impact_at_risk = normalized impressions x normalized decline severity
log_impr = np.log1p(data["gsc_impressions"])
impact_at_risk = (log_impr / log_impr.max()) * (scored["trend_pct_at_T"].abs() / 100)

# --- final priority score ---
scored["p_recovery"] = p_recovery
scored["impact_at_risk"] = impact_at_risk
scored["priority_score"] = scored["impact_at_risk"] * (1 - scored["p_recovery"])

# top quartile impact, bottom quartile recovery -- both genuine minorities, not halves
high_impact = scored["impact_at_risk"] >= scored["impact_at_risk"].quantile(0.75)
low_recovery = scored["p_recovery"] <= scored["p_recovery"].quantile(0.25)

def reason_code(impact_h, recov_l):
    if impact_h and recov_l:
        return "high_impact_low_recovery"
    if impact_h and not recov_l:
        return "high_impact_likely_recovers"
    if not impact_h and recov_l:
        return "low_impact_low_recovery"
    return "low_impact_likely_recovers"

scored["reason_code"] = [
    reason_code(h, r) for h, r in zip(high_impact, low_recovery)
]

scored = scored.sort_values("priority_score", ascending=False).reset_index()
scored.index += 1
print(scored.head(10)[["content_hash_id", "priority_score", "reason_code", "impact_at_risk", "p_recovery"]])

             content_hash_id  priority_score                  reason_code  \
1   content_5519ab199297bdb2        0.527189     high_impact_low_recovery   
2   content_469512b149928d3e        0.471043  high_impact_likely_recovers   
3   content_972690231ea8c69f        0.466190     high_impact_low_recovery   
4   content_eb77135ddda07ea8        0.460199     high_impact_low_recovery   
5   content_c9a819d1b9ad7420        0.456771     high_impact_low_recovery   
6   content_eea3b2434e98edf3        0.455715     high_impact_low_recovery   
7   content_fcc1f8ce43abd284        0.451740     high_impact_low_recovery   
8   content_6945124cd32196d5        0.448963     high_impact_low_recovery   
9   content_4e1a6900719d2ef1        0.441381  high_impact_likely_recovers   
10  content_0f98153d0219fd53        0.439877     high_impact_low_recovery   

    impact_at_risk  p_recovery  
1         0.720535    0.268336  
2         0.859845    0.452177  
3         0.598538    0.221118  
4         0.567226  

In [1]:
scored["reason_code"].value_counts()

NameError: name 'scored' is not defined

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.